# MiliPoint reproduction — training & paper comparison

This notebook reproduces the results of **"MiliPoint: A Point Cloud Dataset for mmWave Radar"**
(Cui, Zhong, Wu, Shen, Dahnoun, Zhao — NeurIPS 2023 Datasets & Benchmarks,
[arXiv:2309.13425](https://arxiv.org/abs/2309.13425)), using the code from
[`yizzfz/MiliPoint`](https://github.com/yizzfz/MiliPoint) (mirrored at
[`KhalidMehebub/Milipoint-reproduction`](https://github.com/KhalidMehebub/Milipoint-reproduction)).

**Before running:**
- Settings → Accelerator → **GPU T4 x2** (or P100).
- Settings → Internet → **On** (needed to `pip install` and clone the repo either way; also needed for the
  Google Drive dataset download if you skip the step below).
- Optional but recommended: Add Input → attach your own uploaded copy of `MiliPoint_data.zip` (or its
  unzipped `.pkl` files) as a Kaggle Dataset, so step 4 uses it directly instead of downloading from Google
  Drive. If you do this, check the Input panel for the actual mount path and set `KAGGLE_INPUT_DIR` in
  step 4 to match.

**What this notebook does:**
1. Installs the `mmrnet` package and its dependencies.
2. Downloads the raw radar point-cloud dataset from Google Drive (`MiliPoint_data.zip`).
3. Trains + evaluates each `(task, model)` combination from the paper's Table 3:
   - Tasks: `mmr_kp_9pt` / `mmr_kp_18pt` (keypoint estimation, 9- and 18-point skeletons), `mmr_iden`
     (person identification), `mmr_act` (action classification)
   - Models: `dgcnn`, `pointtransformer` (paper calls this "Pointformer"), `pointnet` (PointNet++), `pointmlp`
     — plus the repo's plain `mlp` baseline, which is **not** in the paper's Table 3 (no reference number to
     compare it against, trained anyway as a cheap sanity check).
4. Runs each combination over `N_SEEDS` seeds (the paper uses 3, "each data point is run three times with
   different random seeds"; default here is **1**, traded off for time budget — see section 5) and
   aggregates mean ± std.
5. Builds a side-by-side comparison against the paper's actual Table 3 numbers (transcribed directly from
   the paper below, not searched for).

**Time budget:** the paper's own training run took ~300 GPU-hours in total across every task/model/seed on
4×RTX2080Ti / 2×RTX3090Ti systems. A single Kaggle T4 is much slower per-run than that hardware, and Kaggle
sessions cap out at 12 hours (and a weekly GPU quota) — a full 3-seed, 300-epoch, paper-matching sweep will
not fit in one sitting (measured: ~29 hours for just DGCNN on one task at 300 epochs on a T4). Defaults
below are already scaled down (`N_SEEDS=1`, `FULL_RUN_EPOCHS=50`) to something closer to Kaggle-feasible;
use the `QUICK_TEST` toggle to validate the pipeline first, then adjust `TASKS`/`MODELS`/`N_SEEDS`/
`FULL_RUN_EPOCHS` in section 5 based on the real per-epoch timings your own runs report — the loop in
section 7 is resumable, so multi-session runs don't lose progress.


## 1. Environment check

In [ ]:
!nvidia-smi
import torch, sys
print("Python:", sys.version)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| CUDA:", torch.version.cuda)


## 2. Install dependencies

We install `torch_geometric` plus the compiled `torch-scatter` / `torch-sparse` / `torch-cluster` extensions
matched to Kaggle's pre-installed torch + CUDA build (rather than reinstalling torch itself, which risks
breaking the GPU setup). If a prebuilt wheel isn't available for the exact combo, pip falls back to a source
build (slower, a few minutes, but works on Kaggle's CUDA toolkit).

**`torch_geometric` is pinned to `<2.8`.** Every point-based model here (DGCNN, PointNet++,
PointTransformer, PointMLP) calls `fps`/`knn_graph`/`DynamicEdgeConv`, which `torch_geometric>=2.8` only
supports through the separate `pyg-lib` package (which doesn't have prebuilt wheels for every torch/CUDA
combo, and isn't on plain PyPI at all). Versions `<2.8` implement the same ops through `torch-cluster`
instead — the exact dependency the repo's own `readme.md`/`install_mmrnet.sh` already tell you to install —
so pinning below 2.8 avoids `pyg-lib` entirely. (Verified directly: `torch_geometric==2.7.0`'s `fps` and
`DynamicEdgeConv` both call into `torch_cluster`, not `pyg-lib`.)

We also don't pin to the repo's exact `setup.py` versions (`torch_geometric==2.2.0`,
`pytorch_lightning==1.9.3`) — those are old enough to fight with a modern torch install. The one place this
caused an actual incompatibility (`mmrnet`'s `Scale` transform overriding the old `__call__`-based API
instead of `torch_geometric.transforms.BaseTransform`'s current `forward`-based one) is already fixed at the
source in this repo, not runtime-patched.


In [ ]:
import subprocess, sys
import torch


def pip_install(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)


TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() and torch.version.cuda else 'cpu'
PYG_URL = f"https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html"
print("Using wheel index:", PYG_URL)

pip_install("torch_geometric<2.8")

try:
    pip_install("torch-scatter", "torch-sparse", "torch-cluster", "-f", PYG_URL)
except subprocess.CalledProcessError:
    print("Wheel-index install failed, falling back to a source build (slower, needs a CUDA toolchain).")
    # --no-build-isolation: these packages' setup.py does `import torch` at build time to link
    # against libtorch: a normal isolated build env doesn't have torch installed, so it fails.
    pip_install("torch-scatter", "torch-sparse", "torch-cluster", "--no-build-isolation")

pip_install(
    "pytorch_lightning", "toml", "gdown", "pandas", "tqdm",
    "matplotlib", "scikit-learn", "opencv-python-headless",
)


## 3. Get the code

Pinned to the `claude/milipoint-repo-reproduction-mb2hdb` branch: as of writing, PR #1 (which adds the
`mmrnet` code, `mm` CLI, and configs) is still open against `main`, so a plain clone of `main` only gets the
placeholder README and no package to install. Once that PR is merged, change `REPO_BRANCH` to `main`.

Uses `subprocess` rather than `!shell` / `%pip` magics wherever a Python variable is involved — in testing,
Kaggle's IPython did not reliably expand `{python_var}`/`$python_var` inside indented `!` lines, which
silently cloned the wrong branch. `subprocess.run([...])` with a real argument list has no such ambiguity.

Always deletes and re-clones fresh, so re-running this cell can't get stuck on a stale/wrong-branch clone
left over from an earlier attempt. Explicitly `chdir`s to `/kaggle/working` *before* deleting anything: if
this cell has already run once this kernel session, the kernel's cwd is inside `REPO_DIR` (from the
`os.chdir(REPO_DIR)` below) — deleting a directory that is a process's current working directory invalidates
`os.getcwd()` for that process even after the same path is recreated, breaking every subsequent cell
(including `pip install`, which shells out and needs a valid cwd) until the kernel is restarted or something
explicitly `chdir`s elsewhere. Stepping out first avoids that regardless of how many times this cell is
re-run.

`setup.py`'s `install_requires` pulls in a long tail of packages this notebook doesn't need for
training/eval (`deepspeed`, `pyvista`, `wandb`, old pinned `Cython`/`protobuf`/`ninja` versions, ...), some
of which fail to build on Kaggle. `--no-deps` skips all of that — section 2 above already installs every
package actually imported by `mmrnet`'s dataset/model/session code.


In [ ]:
import os, subprocess, sys

REPO_DIR = "/kaggle/working/Milipoint-reproduction"
REPO_BRANCH = "claude/milipoint-repo-reproduction-mb2hdb"  # switch to "main" once PR #1 is merged
REPO_URL = "https://github.com/KhalidMehebub/Milipoint-reproduction"

os.chdir("/kaggle/working")  # never rm -rf a directory we might currently be sitting inside
subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

import mmrnet
print("mmrnet OK, imported from:", mmrnet.__file__)


## 4. Get the dataset

`MiliPoint_data.zip` is normally hosted on Google Drive (linked from the repo's `readme.md`). If you've
attached it to this notebook as a Kaggle Dataset (Add Input), we use that directly — no download needed.
Otherwise we fall back to fetching it from Google Drive with `gdown`.

If you added the dataset as a Kaggle input, set `KAGGLE_INPUT_DIR` below to wherever Kaggle mounted it
(check the Input panel on the right — Kaggle mounts a dataset added as `khalidmehebub/milipoint-data` at
`/kaggle/input/milipoint-data`, so adjust the guessed candidates if yours differs). Both `.pkl` files and
`action_label.npy` (needed for the action classification task) are picked up from whichever candidate
matches — `id.json` doesn't need to come from here, it already ships inside the repo at `data/raw/id.json`.


In [ ]:
import os, glob, shutil, subprocess

DATA_RAW = os.path.join(REPO_DIR, "data", "raw")
os.makedirs(DATA_RAW, exist_ok=True)

# adjust this if your attached dataset mounts somewhere else (see the Input panel)
KAGGLE_INPUT_DIR = "/kaggle/input/datasets/khalidmehebub/milipoint-data"
KAGGLE_INPUT_CANDIDATES = [
    KAGGLE_INPUT_DIR,
    "/kaggle/input/milipoint-data",
]

GDRIVE_FILE_ID = "1rq8yyokrNhAGQryx7trpUqKenDnTI6Ky"  # from the repo's readme.md, used only as a fallback
ZIP_PATH = "/kaggle/working/MiliPoint_data.zip"


def link_or_copy(src_files, dst_dir):
    for f in src_files:
        dst = os.path.join(dst_dir, os.path.basename(f))
        if not os.path.exists(dst):
            try:
                os.symlink(f, dst)
            except OSError:
                shutil.copy2(f, dst)


def count_pkl():
    return len(glob.glob(os.path.join(DATA_RAW, "*.pkl")))


if count_pkl() == 0:
    # 1) look for an already-attached Kaggle input dataset
    for cand in KAGGLE_INPUT_CANDIDATES:
        if not os.path.isdir(cand):
            continue
        pkls = glob.glob(os.path.join(cand, "**", "*.pkl"), recursive=True)
        npys = glob.glob(os.path.join(cand, "**", "*.npy"), recursive=True)
        if pkls:
            print(f"Found {len(pkls)} .pkl files (+{len(npys)} .npy) under {cand}, linking into {DATA_RAW}")
            link_or_copy(pkls + npys, DATA_RAW)
            break
        zips = glob.glob(os.path.join(cand, "**", "*.zip"), recursive=True)
        if zips:
            print(f"Found {zips[0]}, unzipping into {DATA_RAW}")
            subprocess.run(["unzip", "-q", "-o", zips[0], "-d", DATA_RAW], check=True)
            break

if count_pkl() == 0:
    # 2) fall back to downloading from Google Drive (needs internet enabled)
    print("No attached Kaggle dataset found, falling back to Google Drive download.")
    if not os.path.exists(ZIP_PATH):
        subprocess.run(["gdown", "--id", GDRIVE_FILE_ID, "-O", ZIP_PATH], check=True)
    subprocess.run(["unzip", "-q", "-o", ZIP_PATH, "-d", DATA_RAW], check=True)

n_pkl = count_pkl()
print(f"{n_pkl} .pkl files in {DATA_RAW}")
assert n_pkl > 0, (
    "No dataset found. Either attach it as a Kaggle input and fix KAGGLE_INPUT_DIR above, "
    "or enable internet so the Google Drive fallback can run."
)


## 5. Experiment grid

Taken from the paper's Section 4.1 ("Experiment Setup") and Table 3/4, **not** the repo's CLI defaults,
which differ in two places:

| | paper | repo CLI default |
|---|---|---|
| learning rate | `3e-5` (Adam + CosineAnnealing) | `1e-5` |
| action classification stacking `s` | `50` | the shipped config (`mmr_action_stack_5_point.toml`) uses `5` |

Everything else lines up: Adam optimizer, batch size 128, 0.8/0.1/0.1 split, per-data-point zero padding,
`s=5` for identification and both keypoint variants. `mmrnet`'s `ModelWrapper` already implements
Adam+CosineAnnealingLR to match the paper, so only `learning_rate` and the action task's `stacks` need
overriding here.

Each keypoint/action override gets its own cached-data filename (via `processed_data_suffix` below) so it
doesn't collide with the repo's default `stack_5` cache for the same task.

Set `QUICK_TEST = True` first to confirm every `(task, model)` pair trains and evaluates end-to-end with a
tiny epoch/seed budget, before switching to `False` for the full run.

**Time budget, measured, not guessed:** a real Kaggle T4 run of DGCNN on `mmr_kp_9pt` (batch size 128,
~3,400 train batches/epoch) logged **~5:47/epoch**. At the paper's own `max_epochs=300` that's **~29 hours
for one single (task, model, seed)** — for the full 4 task × 5 model × 3 seed grid (60 runs) that's
somewhere around 1,700+ GPU-hours, nowhere near Kaggle's session/quota limits. So below, **`N_SEEDS=1`**
(instead of the paper's 3 — you lose the paper's mean±std averaging, reproduced numbers become single-run
point estimates) and **`FULL_RUN_EPOCHS=50`** (instead of 300 — ~4:50 for that same DGCNN/`mmr_kp_9pt`
combination, scale the rest from there). `mmr_act` uses 10x more points per sample than the other tasks
(`stacks=50` vs `5`, at `max_points=22` each), so expect it to run considerably slower per batch than this
estimate suggests — time it separately before assuming the same budget applies. Adjust `FULL_RUN_EPOCHS` /
`N_SEEDS` / `MODELS` once you've seen a couple of real per-epoch timings from your own runs.


In [ ]:
QUICK_TEST = True   # flip to False for the full run

TASKS = {
    "mmr_kp_9pt":  {
        "dataset": "mmr_kp",
        "config": "configs/keypoints/mmr_keypoints_stack_5_point.toml",
        "overrides": {"num_keypoints": 9},
        "processed_data_suffix": "_9pt",
    },
    "mmr_kp_18pt": {
        "dataset": "mmr_kp",
        "config": "configs/keypoints/mmr_keypoints_stack_5_point.toml",
        "overrides": {"num_keypoints": 18},
        "processed_data_suffix": "_18pt",
    },
    "mmr_iden": {
        "dataset": "mmr_iden",
        "config": "configs/iden/mmr_iden_stack_5_point.toml",
        "overrides": {},
        "processed_data_suffix": "",
    },
    "mmr_act": {
        "dataset": "mmr_act",
        "config": "configs/action/mmr_action_stack_5_point.toml",
        "overrides": {"stacks": 50},   # paper uses s=50 for action, the shipped config uses 5
        "processed_data_suffix": "_stack50",
    },
}
MODELS = ["dgcnn", "pointtransformer", "pointnet", "pointmlp", "mlp"]

N_SEEDS = 1            # paper uses 3 different seeds and reports mean +/- std; traded off here for feasibility
FULL_RUN_EPOCHS = 50   # paper's own examples use 300; traded off here for feasibility -- see the time-budget note above

TRAIN_CONFIG = dict(
    optimizer="adam",
    learning_rate=3e-5,     # paper Section 4.1, not the repo CLI default of 1e-5
    weight_decay=1e-5,      # not specified by the paper; kept at the repo's default
    batch_size=128,
    max_epochs=(3 if QUICK_TEST else FULL_RUN_EPOCHS),
    num_workers=2,
)
SEEDS = [20] if QUICK_TEST else [20 + i for i in range(N_SEEDS)]

print("Grid:", [(t, m) for t in TASKS for m in MODELS])
print("Seeds:", SEEDS)
print("Train config:", TRAIN_CONFIG)


## 6. Train + evaluate helper

Calls the same `mmrnet.session.train.train` / dataset / model code the CLI (`./mm train ...` / `./mm eval ...`)
uses, but in-process so we can capture the returned test metrics directly instead of scraping stdout.


In [ ]:
import base64, os
import gc, time, random, logging
import toml as toml_lib
import numpy as np
import torch
import pytorch_lightning as pl
from IPython.display import HTML, display

from mmrnet.dataset import get_dataset
from mmrnet.models import model_map
from mmrnet.session.train import train as mm_train
from mmrnet.session.wrapper import ModelWrapper

logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)

RESULTS_CSV = "/kaggle/working/reproduced_results_raw.csv"


def trigger_browser_download(path):
    """Push a file straight to the browser's downloads, no click needed.

    Kaggle's Output tab already keeps everything under /kaggle/working/ after
    the notebook finishes, but that requires committing and waiting for a
    save -- this gets the file onto your machine the moment a cell finishes,
    while the session is still live. If your browser blocks the download
    (some pop-up/download blockers interfere with programmatic clicks),
    grab the file from the Output tab instead, or from the file browser on
    the left (Kaggle inserts a manual download button next to each file
    there too).
    """
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    filename = os.path.basename(path)
    link_id = f"dl_{abs(hash(path + str(time.time())))}"
    display(HTML(f'''
        <a id="{link_id}" download="{filename}" href="data:text/csv;base64,{b64}"></a>
        <script>document.getElementById("{link_id}").click();</script>
    '''))
    print(f"Triggered browser download for {filename}")


def build_dataset_config(task_key):
    task_cfg = TASKS[task_key]
    with open(task_cfg["config"]) as f:
        cfg = toml_lib.load(f)
    cfg.update(task_cfg["overrides"])
    if task_cfg["processed_data_suffix"]:
        base, ext = cfg["processed_data"].rsplit(".", 1)
        cfg["processed_data"] = f"{base}{task_cfg['processed_data_suffix']}.{ext}"
    return task_cfg["dataset"], cfg


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    np.random.seed(seed)


def run_one(task_key, model_name, seed):
    set_seed(seed)
    dataset_name, mmr_dataset_config = build_dataset_config(task_key)

    train_loader, val_loader, test_loader, info = get_dataset(
        name=dataset_name,
        batch_size=TRAIN_CONFIG["batch_size"],
        workers=TRAIN_CONFIG["num_workers"],
        mmr_dataset_config=mmr_dataset_config,
    )

    model = model_map[model_name](info=info)

    plt_trainer_args = {
        "max_epochs": TRAIN_CONFIG["max_epochs"],
        "devices": 1,
        "accelerator": "gpu" if torch.cuda.is_available() else "cpu",
        "strategy": "auto",   # modern pytorch_lightning rejects strategy=None
        "fast_dev_run": False,
        "enable_progress_bar": False,
        "logger": False,
    }

    save_path = f"checkpoints/{task_key}_{model_name}_seed{seed}"
    t0 = time.time()
    mm_train(
        model=model, train_loader=train_loader, val_loader=val_loader,
        optimizer=TRAIN_CONFIG["optimizer"], learning_rate=TRAIN_CONFIG["learning_rate"],
        weight_decay=TRAIN_CONFIG["weight_decay"], plt_trainer_args=plt_trainer_args,
        save_path=save_path,
    )
    train_seconds = time.time() - t0

    # reload best checkpoint and evaluate on the held-out test split, mirroring mmrnet/session/test.py
    eval_model = ModelWrapper(model_map[model_name](info=info))
    state_dict = torch.load(f"{save_path}/best.ckpt")["state_dict"]
    eval_model.load_state_dict(state_dict)
    eval_model.eval()

    test_trainer = pl.Trainer(
        devices=1, accelerator="gpu" if torch.cuda.is_available() else "cpu",
        enable_progress_bar=False, logger=False,
    )
    test_results = test_trainer.test(eval_model, test_loader, verbose=False)[0]

    del model, eval_model, train_loader, val_loader, test_loader
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "task": task_key, "model": model_name, "seed": seed,
        "train_seconds": round(train_seconds, 1), **test_results,
    }


## 7. Run the sweep

Resumable: if `reproduced_results_raw.csv` already has a row for a `(task, model, seed)` triple, it's
skipped, so a Kaggle session timeout doesn't lose earlier progress — just re-run this cell after restarting.

When the cell finishes, `trigger_browser_download` pushes `reproduced_results_raw.csv` straight to your
browser's downloads automatically (no click needed) — same for the aggregated and paper-comparison CSVs at
the end of sections 8 and 10. If your browser blocks it (pop-up/download blockers can interfere with a
programmatic click), the files are still sitting in `/kaggle/working/` and downloadable from there.


In [ ]:
import pandas as pd
import os

if os.path.exists(RESULTS_CSV):
    raw_df = pd.read_csv(RESULTS_CSV)
else:
    raw_df = pd.DataFrame()

done = set(zip(raw_df.get("task", []), raw_df.get("model", []), raw_df.get("seed", [])))

for task_key in TASKS:
    for model_name in MODELS:
        for seed in SEEDS:
            if (task_key, model_name, seed) in done:
                print(f"skip {task_key}/{model_name}/seed{seed} (already have a result)")
                continue
            print(f"=== training {task_key}/{model_name}/seed{seed} ===")
            try:
                row = run_one(task_key, model_name, seed)
            except Exception as e:
                print(f"FAILED {task_key}/{model_name}/seed{seed}: {e}")
                continue
            print(row)
            raw_df = pd.concat([raw_df, pd.DataFrame([row])], ignore_index=True)
            raw_df.to_csv(RESULTS_CSV, index=False)

trigger_browser_download(RESULTS_CSV)
raw_df


## 8. Aggregate across seeds

- Classification tasks (`mmr_iden`, `mmr_act`) report `test_acc` (Top-1) and `test_top3_acc`.
- Keypoint tasks report `test_mle` (mean localization error). The dataset applies a `x100` scale transform
  to keypoint coordinates, so `test_mle` is already in the same units (cm) the paper reports.
- Aggregated as mean ± std across `SEEDS`, matching how the paper reports Table 3.


In [ ]:
agg_rows = []
for (task_key, model_name), grp in raw_df.groupby(["task", "model"]):
    row = {"task": task_key, "model": model_name, "n_seeds": len(grp)}
    if "test_acc" in grp:
        row["acc_%_mean"] = round((grp["test_acc"] * 100).mean(), 2)
        row["acc_%_std"] = round((grp["test_acc"] * 100).std(ddof=0), 2)
    if "test_top3_acc" in grp:
        row["top3_acc_%_mean"] = round((grp["test_top3_acc"] * 100).mean(), 2)
        row["top3_acc_%_std"] = round((grp["test_top3_acc"] * 100).std(ddof=0), 2)
    if "test_mle" in grp:
        row["mle_cm_mean"] = round(grp["test_mle"].mean(), 2)
        row["mle_cm_std"] = round(grp["test_mle"].std(ddof=0), 2)
    agg_rows.append(row)

agg_df = pd.DataFrame(agg_rows).sort_values(["task", "model"])
AGG_CSV = "/kaggle/working/reproduced_results_aggregated.csv"
agg_df.to_csv(AGG_CSV, index=False)
trigger_browser_download(AGG_CSV)
agg_df


## 9. Paper's Table 3, transcribed directly

These values are transcribed from **Table 3** of the paper (page 7 of the NeurIPS PDF) — not searched for —
so they should be exact. Model name mapping: the paper's "Pointformer" is this repo's `pointtransformer`;
"PointNet++" is `pointnet`. The paper's Table 3 has **no plain-MLP row** — only `Random`, `DGCNN`,
`Pointformer`, `PointNet++`, `PointMLP` — so there's nothing to compare the repo's `mlp` baseline against.


In [ ]:
# (task, model) -> paper's reported mean +/- std, from Table 3
paper_results = {
    ("mmr_iden", "dgcnn"):            {"metric": "acc_%",      "mean": 77.65, "std": 0.92},
    ("mmr_iden", "pointtransformer"): {"metric": "acc_%",      "mean": 83.94, "std": 0.81},
    ("mmr_iden", "pointnet"):         {"metric": "acc_%",      "mean": 87.30, "std": 0.27},
    ("mmr_iden", "pointmlp"):         {"metric": "acc_%",      "mean": 95.88, "std": 0.40},

    ("mmr_act", "dgcnn"):             {"metric": "top1_%",     "mean": 13.61, "std": 2.09, "top3_mean": 34.59, "top3_std": 2.74},
    ("mmr_act", "pointtransformer"):  {"metric": "top1_%",     "mean": 29.27, "std": 0.55, "top3_mean": 50.44, "top3_std": 1.18},
    ("mmr_act", "pointnet"):          {"metric": "top1_%",     "mean": 34.45, "std": 0.80, "top3_mean": 54.96, "top3_std": 1.21},
    ("mmr_act", "pointmlp"):          {"metric": "top1_%",     "mean": 18.37, "std": 0.08, "top3_mean": 35.94, "top3_std": 0.14},

    ("mmr_kp_9pt", "dgcnn"):            {"metric": "mle_cm", "mean": 16.53, "std": 0.11},
    ("mmr_kp_9pt", "pointtransformer"): {"metric": "mle_cm", "mean": 14.99, "std": 0.03},
    ("mmr_kp_9pt", "pointnet"):         {"metric": "mle_cm", "mean": 13.55, "std": 0.03},
    ("mmr_kp_9pt", "pointmlp"):         {"metric": "mle_cm", "mean": 13.12, "std": 0.30},

    ("mmr_kp_18pt", "dgcnn"):            {"metric": "mle_cm", "mean": 18.51, "std": 0.03},
    ("mmr_kp_18pt", "pointtransformer"): {"metric": "mle_cm", "mean": 17.03, "std": 0.13},
    ("mmr_kp_18pt", "pointnet"):         {"metric": "mle_cm", "mean": 14.94, "std": 0.03},
    ("mmr_kp_18pt", "pointmlp"):         {"metric": "mle_cm", "mean": 14.11, "std": 0.22},
}

# Table 3's "Random" (untrained weights) baseline row, for context only -- not something this notebook trains.
paper_random_baseline = {
    "iden_acc_%": 7.69,
    "action_top1_%": 2.59, "action_top3_%": 7.69,
    "kp_9pt_mle_cm": 155.74, "kp_9pt_mle_cm_std": 1.32,
    "kp_18pt_mle_cm": 161.64, "kp_18pt_mle_cm_std": 2.11,
}
print("Random baseline (paper, for context):", paper_random_baseline)


## 10. Side-by-side comparison

In [ ]:
rows = []
for _, r in agg_df.iterrows():
    key = (r["task"], r["model"])
    paper = paper_results.get(key)
    if paper is None:
        rows.append({"task": r["task"], "model": r["model"], "metric": "n/a (no paper row)",
                      "reproduced": None, "paper": None, "delta": None})
        continue

    if paper["metric"] == "acc_%":
        reproduced, target = r.get("acc_%_mean"), paper["mean"]
    elif paper["metric"] == "top1_%":
        reproduced, target = r.get("acc_%_mean"), paper["mean"]
    elif paper["metric"] == "mle_cm":
        reproduced, target = r.get("mle_cm_mean"), paper["mean"]
    else:
        reproduced, target = None, None

    rows.append({
        "task": r["task"], "model": r["model"], "metric": paper["metric"],
        "reproduced": reproduced, "paper": target, "paper_std": paper["std"],
        "delta": None if reproduced is None or target is None else round(reproduced - target, 2),
    })

comparison_df = pd.DataFrame(rows).sort_values(["task", "model"])
COMPARISON_CSV = "/kaggle/working/reproduced_vs_paper.csv"
comparison_df.to_csv(COMPARISON_CSV, index=False)
trigger_browser_download(COMPARISON_CSV)
comparison_df


## Notes on interpreting differences

- `QUICK_TEST = True` only trains for a few epochs on 1 seed — do **not** compare those numbers to the
  paper, they're only there to confirm the pipeline runs end-to-end. Set `QUICK_TEST = False` and re-run
  sections 5, 7, 8 for real numbers.
- With `N_SEEDS=1` (traded off from the paper's 3 for time budget — see the note in section 5),
  `agg_df`/`comparison_df`'s `_std` columns are meaningless (std of one run is always 0) — the reproduced
  values are single-run point estimates, not the mean±std the paper reports. Differences that would fall
  within the paper's own seed-to-seed variance can't be distinguished from real deviations with only 1 seed.
- `FULL_RUN_EPOCHS=50` here is also a time-budget compromise, not a paper-matched number — the paper doesn't
  state an exact epoch count, only that the full sweep (all tasks × models × 3 seeds) took ~300 GPU-hours on
  4×RTX2080Ti / 2×RTX3090Ti, and the repo's own example commands in `readme.md` use 300. A single Kaggle T4
  measured at ~5:47/epoch for DGCNN on `mmr_kp_9pt` makes 300 epochs alone ~29 hours per run — far outside
  Kaggle's session/quota limits for the full grid. `train_seconds` in the raw results shows the real time
  each run took, so you can tell whether a run was cut short or just genuinely undertrained relative to 300.
- Small remaining differences from library-version drift (PyTorch/PyG versions differ from whatever the
  paper's authors used in 2023) or plain run-to-run variance are expected even with everything above matched.
